# Agent Human-in-the-Loop: Interrupts, Approval Gates, and Editable State

This notebook is a companion to `agent_memory_deepdive.ipynb` and
`agent_context_engineering.ipynb` in this same folder. It goes deep on one
topic those notebooks only touch in passing: **how to safely pause an
agent mid-execution for a human, and resume it exactly where it left off.**

By the end you will have built, with real code against a real LLM API:

1. **Three distinct HITL mechanisms** in LangGraph, and a clear decision
   rule for which one fits which situation.
2. **An approval gate** on a mutating action (refund) — pause, get a
   yes/no, resume.
3. **Editable state** — the human doesn't just approve/reject, they correct
   a value before the agent continues.
4. **Conditional interrupts** — only pausing for a human when a risk
   threshold is actually crossed, with a configurable, tunable threshold.
5. **A combined finale** — one small agent where a low-risk case sails
   through untouched and a high-risk case pauses, gets edited, and resumes.

## Prerequisites

This notebook assumes you've already seen `agent_memory_deepdive.ipynb`,
specifically the **"Interrupt + resume, still within `InMemorySaver`"**
section. That section proved the mechanism works; this notebook is about
**when and why to reach for it, and the three different shapes it can
take.** HITL literally cannot function without a checkpointer — the graph
has nowhere to persist its paused state without one — so if you haven't
seen that notebook's Part 2, read that first.

## Why this lives here, not inside `multi_agent_architectures/`

Human-in-the-loop is a **horizontal** concept: it applies identically
whether you're running a single ReAct agent, a supervisor, a
planner-executor, or any other topology in this repo's multi-agent
series. It's not a topology itself — it's a control you can drop into
*any* of them, the same way memory and context engineering are properties
of an agent's design rather than architectures in their own right. That's
why it sits alongside those two notebooks instead of getting its own
numbered slot in `multi_agent_architectures/`.

## Setup

In [1]:
import os
import warnings
import logging
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

# Load the repo-root .env (same convention as notebook.ipynb / agent_memory_deepdive.ipynb)
load_dotenv("../../.env")

# Single visible flag controlling which provider the whole notebook uses --
# same convention as the sibling notebooks in this folder. No silent
# auto-detection: the matching key must be present in .env.
PROVIDER = "openai"  # or "anthropic"

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model="claude-sonnet-5", api_key=key)
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.3, api_key=key)
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [block["text"] for block in content if isinstance(block, dict) and block.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o-mini


## Part 1 -- Theory: three different HITL mechanisms, and when each one fits

"Human-in-the-loop" is not one technique -- it's a family of three, and
picking the wrong one is the most common HITL mistake: either pausing
everywhere out of caution (killing throughput) or hand-rolling a fragile
approval flow when LangGraph already has a primitive for it.

| Mechanism | What it is | Use it when | Don't use it when |
|---|---|---|---|
| **`interrupt_before=[node]`** (static, set at `.compile()`) | The graph *always* pauses immediately before a named node runs, no exceptions, no custom payload | You want an unconditional checkpoint before a specific node, every single time, and a plain "should I continue?" is enough | You need to gate *conditionally* (only sometimes) or show the human something specific to review |
| **`interrupt()`** (dynamic, called from inside a node) | The node itself decides, at runtime, whether to pause and what payload to surface | The decision to pause is conditional (risk score, dollar amount, confidence threshold) or the human needs to see/edit a specific value, not just say yes/no | The gate is unconditional and payload-free -- static `interrupt_before` is simpler and does the same job with less code |
| **Return-draft, separate commit call** (no LangGraph primitive at all) | The graph just stops after producing a draft; a *second*, completely separate function call is what actually executes the action | Simple stateless request/response APIs, no checkpointer in use, or the "pause" might last so long (days) that keeping a graph paused isn't the right mental model anyway | You need the *same* graph run to actually resume mid-execution -- this pattern re-runs nothing, it just never continues automatically |

The rest of this notebook builds Parts 2-4 as three concrete, runnable
examples of when each row of this table is the right call -- not as three
interchangeable ways to do the same thing.

```text
                    Does the pause depend on
                    runtime data (amount, risk,
                    confidence)?
                           |
                 +---------+---------+
                 | NO                | YES
                 v                   v
      interrupt_before=[node]     interrupt() inside the node
      (Part 2 uses this shape)    (Parts 3 and 4 use this shape)

      Do you even have a
      checkpointer / long-running
      graph process?
                 |
                 v
               NO --> return-draft-then-separate-commit-call
                      (not built as a full example here --
                      it's a "no LangGraph needed" escape hatch,
                      not a LangGraph pattern)
```

## Part 2 -- Approval gate on a mutating action

**The use case**: a refund-processing agent. Refunds are exactly the kind
of action this notebook's Part 1 table says should get an **unconditional**
gate to start with -- money moving out of the business, hard to reverse,
no reason to skip the check "just this once." This directly echoes the
un-gated `issue_refund` risk shown in
`multi_agent_architectures/04_planner_executor.ipynb`, where a bare tool
call could fire with no separated approval step at all.

### Theory: without HITL vs. with HITL

**Without HITL**: the tool fires the moment the agent decides a refund is
warranted -- there is no artifact a human could review *before* money
moves, only a log to inspect *after the fact*, when it's too late to stop
it.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

_REFUNDS_ISSUED = []


def issue_refund(order_id: str, amount: float) -> str:
    _REFUNDS_ISSUED.append((order_id, amount))
    return f"Refund of ${amount:.2f} issued for order {order_id}."


class NoHitlState(TypedDict):
    order_id: str
    amount: float
    outcome: str


def draft_and_fire_refund(state: NoHitlState) -> dict:
    # No pause, no artifact for a human to see before this executes.
    result = issue_refund(state["order_id"], state["amount"])
    return {"outcome": result}


no_hitl_builder = StateGraph(NoHitlState)
no_hitl_builder.add_node("fire", draft_and_fire_refund)
no_hitl_builder.add_edge(START, "fire")
no_hitl_builder.add_edge("fire", END)
no_hitl_graph = no_hitl_builder.compile()

result = no_hitl_graph.invoke({"order_id": "ORD-9001", "amount": 249.99, "outcome": ""})
print(result["outcome"])
print("Refunds issued so far (no human ever saw this coming):", _REFUNDS_ISSUED)


Refund of $249.99 issued for order ORD-9001.
Refunds issued so far (no human ever saw this coming): [('ORD-9001', 249.99)]


**With HITL**: `interrupt()` pauses the graph immediately before
`issue_refund` runs, surfaces the proposed action, and only calls the tool
once a human resumes with an approval.

In [3]:
from langgraph.types import interrupt, Command

_REFUNDS_ISSUED_GATED = []


class GatedState(TypedDict):
    order_id: str
    amount: float
    approved: bool
    outcome: str


def request_approval(state: GatedState) -> dict:
    # interrupt() itself is the entire "pause and ask" step -- nothing before
    # it in this node has any side effect, which matters (see Common errors).
    decision = interrupt({
        "action": "issue_refund",
        "order_id": state["order_id"],
        "amount": state["amount"],
        "question": f"Approve a ${state['amount']:.2f} refund for order {state['order_id']}?",
    })
    return {"approved": decision}


def act_on_decision(state: GatedState) -> dict:
    if state["approved"]:
        _REFUNDS_ISSUED_GATED.append((state["order_id"], state["amount"]))
        return {"outcome": issue_refund(state["order_id"], state["amount"])}
    return {"outcome": "Refund rejected by reviewer -- no action taken."}


gated_builder = StateGraph(GatedState)
gated_builder.add_node("approval", request_approval)
gated_builder.add_node("act", act_on_decision)
gated_builder.add_edge(START, "approval")
gated_builder.add_edge("approval", "act")
gated_builder.add_edge("act", END)
gated_graph = gated_builder.compile(checkpointer=InMemorySaver())

gated_config = {"configurable": {"thread_id": "refund-approval-1"}}
paused = gated_graph.invoke({"order_id": "ORD-9002", "amount": 249.99, "approved": False, "outcome": ""}, config=gated_config)
print("Paused. Interrupt payload for the human:", paused["__interrupt__"])
print("Refunds issued so far (should be empty -- nothing executed yet):", _REFUNDS_ISSUED_GATED)


Paused. Interrupt payload for the human: [Interrupt(value={'action': 'issue_refund', 'order_id': 'ORD-9002', 'amount': 249.99, 'question': 'Approve a $249.99 refund for order ORD-9002?'}, id='f68422f2833a3d2e99684a42b123f6e6')]
Refunds issued so far (should be empty -- nothing executed yet): []


In [4]:
# A human reviews the payload above and approves it. Resume on the SAME thread_id.
resumed = gated_graph.invoke(Command(resume=True), config=gated_config)
print("Resumed outcome:", resumed["outcome"])
print("Refunds issued (now populated, only after explicit approval):", _REFUNDS_ISSUED_GATED)


Resumed outcome: Refund of $249.99 issued for order ORD-9002.
Refunds issued (now populated, only after explicit approval): [('ORD-9002', 249.99)]


### Mermaid

```mermaid
graph LR
    START([START]) --> approval[approval node<br/>calls interrupt, no side effects]
    approval -.pauses here,<br/>state checkpointed.-> approval
    approval --> act[act node<br/>only node that touches issue_refund]
    act --> END([END])
```

### When to use this exact shape

Use an unconditional `interrupt()` gate (or the even simpler static
`interrupt_before=["act"]`, if you don't need a custom payload) any time
the action is irreversible or costly enough that **every** instance
deserves a human step, regardless of how confident the agent is --
payments, deletions, sending external communications, filing anything
regulatory. If you find yourself wanting to skip the gate "when the
agent is really sure," that's actually Part 4 (conditional interrupts),
not a reason to remove this one.

### Common errors

- **No checkpointer bound to `.compile()`.** Without one, there's nowhere
  to persist the paused state -- `interrupt()` has nothing to resume
  *from*, and `Command(resume=...)` has nothing to attach to.
- **Side effects placed *before* `interrupt()` inside the same node.**
  This is the single most damaging HITL bug, and it's easy to miss: when a
  node containing `interrupt()` is resumed, LangGraph re-runs the **entire
  node function from the top** -- not just the code after the interrupt
  call. Any API call, log write, or state mutation placed before
  `interrupt()` in that node will fire **again** on resume. That's exactly
  why `request_approval` above does nothing but build the payload and call
  `interrupt()` -- the actual `issue_refund` call lives in a separate
  node (`act`), which only ever runs once, after resume.
- **Resuming with the wrong payload shape.** `Command(resume=True)` here
  becomes `decision` inside `request_approval`; if `act_on_decision`
  expects a dict and gets a bare boolean (or vice versa), you get a
  `KeyError`/`TypeError` deep in a node with no obvious link back to the
  resume call that caused it. Keep the resume payload's shape and the
  node's expectation of it in sync deliberately, not by convention.

## Part 3 -- Editable state, not just approve/reject

**The use case**: the same refund flow, but this time the agent
*extracted* the refund amount from a customer email, and extraction can
be wrong. A binary approve/reject throws away a request that's 90%
right -- what you actually want is for the human to **correct** the
amount and let the agent continue with the corrected value, not restart
the whole conversation.

### Theory

This is a different shape from Part 2's gate: instead of `interrupt()`
returning a boolean that branches the graph, it returns a **value that
overwrites part of state** and flows into everything downstream, exactly
as if the agent itself had produced that corrected value.

In [5]:
from pydantic import BaseModel, Field

CUSTOMER_EMAIL = (
    "Hi, I need a refund for order ORD-9003. My statement shows an initial "
    "authorization hold of $1,800.00 that never got released, and the "
    "actual charge that went through for $180.00. Please refund me for "
    "this order. Thanks."
)


class ExtractedAmount(BaseModel):
    amount: float = Field(description="The refund amount the customer is owed, in dollars")


extract_llm = llm.with_structured_output(ExtractedAmount)


class EditableState(TypedDict):
    order_id: str
    extracted_amount: float
    final_amount: float
    outcome: str


def extract_amount(state: EditableState) -> dict:
    # A real LLM extraction call, deliberately fed an email with TWO dollar amounts --
    # a released authorization hold ($1,800, not actually owed) and the real charge
    # ($180, the correct refund). Genuinely ambiguous, not scripted to fail --
    # whichever way the real model reads it is what Part 3 demonstrates correcting
    # (or confirming) below.
    result = extract_llm.invoke(f"Extract the refund amount the customer is owed from this email:\n\n{CUSTOMER_EMAIL}")
    return {"extracted_amount": result.amount}


def human_review_amount(state: EditableState) -> dict:
    corrected = interrupt({
        "action": "review_extracted_amount",
        "order_id": state["order_id"],
        "extracted_amount": state["extracted_amount"],
        "question": "Is this the correct refund amount? Provide the correct value if not.",
    })
    return {"final_amount": corrected}


def issue_corrected_refund(state: EditableState) -> dict:
    return {"outcome": issue_refund(state["order_id"], state["final_amount"])}


edit_builder = StateGraph(EditableState)
edit_builder.add_node("extract", extract_amount)
edit_builder.add_node("review", human_review_amount)
edit_builder.add_node("act", issue_corrected_refund)
edit_builder.add_edge(START, "extract")
edit_builder.add_edge("extract", "review")
edit_builder.add_edge("review", "act")
edit_builder.add_edge("act", END)
edit_graph = edit_builder.compile(checkpointer=InMemorySaver())

edit_config = {"configurable": {"thread_id": "refund-edit-1"}}
paused = edit_graph.invoke({"order_id": "ORD-9003", "extracted_amount": 0.0, "final_amount": 0.0, "outcome": ""}, config=edit_config)
print("Paused for review. Extracted amount:", paused["__interrupt__"])


Paused for review. Extracted amount: [Interrupt(value={'action': 'review_extracted_amount', 'order_id': 'ORD-9003', 'extracted_amount': 180.0, 'question': 'Is this the correct refund amount? Provide the correct value if not.'}, id='685aebbc01a37ace5b234ff6d1549a5e')]


In [6]:
# The correct refund is the $180.00 charge that actually went through -- the $1,800
# was only an authorization hold, never an actual charge, so it is not refundable.
# Whatever the extraction step above produced, the human now confirms or corrects it
# to the one number that is actually right.
resumed = edit_graph.invoke(Command(resume=180.00), config=edit_config)
print("Extraction step originally produced:", paused["__interrupt__"])
print("Resumed outcome, using the human-confirmed-or-corrected amount:", resumed["outcome"])


Extraction step originally produced: [Interrupt(value={'action': 'review_extracted_amount', 'order_id': 'ORD-9003', 'extracted_amount': 180.0, 'question': 'Is this the correct refund amount? Provide the correct value if not.'}, id='685aebbc01a37ace5b234ff6d1549a5e')]
Resumed outcome, using the human-confirmed-or-corrected amount: Refund of $180.00 issued for order ORD-9003.


**Expected output, and what actually happened on this run**: this was a real,
non-scripted extraction call over a genuinely ambiguous email (two dollar amounts,
one a red herring), not one rigged to fail. On this run, `gpt-4o-mini` **correctly**
extracted $180.00 -- the actual charge, correctly distinguishing it from the $1,800
authorization hold that was never actually charged. That means this run demonstrates
the **confirmation** case, not the **correction** case: the human's resume value is
the same $180.00, re-affirming a right answer rather than fixing a wrong one.

That is still a legitimate, honest result worth keeping rather than editing toward a
cleaner-looking "gotcha" -- a human sign-off step doesn't stop being useful just
because the extraction happened to be right, and this notebook isn't going to claim
a failure that didn't occur. If you want to see the **correction** case fire
instead, re-run cell 11 a few times (extraction is non-deterministic across calls)
or make `CUSTOMER_EMAIL` more ambiguous still -- the mechanism itself doesn't
change either way: `act`'s code is identical between Part 2 and here, and it never
needs to know or care whether the value in state came from the agent's first guess
or a human's override -- `interrupt()`'s return value is indistinguishable from a
value the agent computed itself.

### When to use this shape

Use editable-state `interrupt()` any time the realistic space of human responses is
broader than yes/no -- correcting extracted data, editing AI-drafted customer-facing
text, adjusting a computed value. If "reject and let the agent try again from
scratch" is an acceptable fallback, Part 2's simpler approve/reject gate is enough;
reach for this the moment "reject" would waste a customer's time or the agent's
context on a mostly-right answer that just needed one field fixed.

### Common errors

- **Trusting the edited value without re-validating it.** HITL is not a substitute
  for input validation -- a human can type a negative amount, an empty string, or a
  value with a typo just as easily as an LLM can misextract one. `act` above skips
  validation for brevity; a real system needs the same amount/type checks after
  `interrupt()` that it would need on any other untrusted input.
- **Assuming the human always has full context.** The payload handed to
  `interrupt()` is the *only* information the reviewer sees unless your UI fetches
  more -- if `extracted_amount` alone isn't enough context to judge correctness
  (as in this real example, where the email's *two* dollar amounts matter), the
  payload needs to carry that context too, not just the field being corrected.

## Part 4 -- Conditional interrupts: not every action needs a human

**The use case**: the same refund agent, but now at realistic volume.
Reviewing 100% of refunds by hand isn't operationally viable once a
system processes thousands of them -- and it defeats much of the point of
automating the workflow in the first place. Most production HITL systems
gate only the **risky tail** of actions, not every single one.

### Theory

The interrupt itself doesn't change -- what changes is a plain `if` check
*before* deciding to call it, based on a **configurable** threshold, the
same "expose the tunable knob, don't hardcode it" discipline used for
`PIPELINE_CONFIG["max_steps"]` in the planner-executor multi-agent
notebook.

In [7]:
# Configurable, not hardcoded -- tune this per how expensive a wrong auto-approval is.
HITL_CONFIG = {"auto_approve_below": 50.00}


class ConditionalState(TypedDict):
    order_id: str
    amount: float
    approved: bool
    outcome: str


def maybe_request_approval(state: ConditionalState) -> dict:
    if state["amount"] < HITL_CONFIG["auto_approve_below"]:
        # Below the threshold: no interrupt() call at all. This branch never pauses.
        return {"approved": True}
    decision = interrupt({
        "action": "issue_refund", "order_id": state["order_id"], "amount": state["amount"],
        "question": f"Refund of ${state['amount']:.2f} exceeds the auto-approve threshold (${HITL_CONFIG['auto_approve_below']:.2f}). Approve?",
    })
    return {"approved": decision}


def act_conditional(state: ConditionalState) -> dict:
    if state["approved"]:
        return {"outcome": issue_refund(state["order_id"], state["amount"])}
    return {"outcome": "Refund rejected by reviewer -- no action taken."}


cond_builder = StateGraph(ConditionalState)
cond_builder.add_node("maybe_approval", maybe_request_approval)
cond_builder.add_node("act", act_conditional)
cond_builder.add_edge(START, "maybe_approval")
cond_builder.add_edge("maybe_approval", "act")
cond_builder.add_edge("act", END)
cond_graph = cond_builder.compile(checkpointer=InMemorySaver())

# Low-risk case: under the threshold, runs straight through, no pause at all.
low_risk_config = {"configurable": {"thread_id": "refund-cond-low"}}
low_risk_result = cond_graph.invoke({"order_id": "ORD-9004", "amount": 12.50, "approved": False, "outcome": ""}, config=low_risk_config)
print("Low-risk ($12.50): ran straight through ->", low_risk_result["outcome"])

# High-risk case: over the threshold, pauses for a human exactly like Part 2.
high_risk_config = {"configurable": {"thread_id": "refund-cond-high"}}
high_risk_paused = cond_graph.invoke({"order_id": "ORD-9005", "amount": 340.00, "approved": False, "outcome": ""}, config=high_risk_config)
print("High-risk ($340.00): paused ->", high_risk_paused.get("__interrupt__", "(did not pause)"))


Low-risk ($12.50): ran straight through -> Refund of $12.50 issued for order ORD-9004.
High-risk ($340.00): paused -> [Interrupt(value={'action': 'issue_refund', 'order_id': 'ORD-9005', 'amount': 340.0, 'question': 'Refund of $340.00 exceeds the auto-approve threshold ($50.00). Approve?'}, id='7619374186d0abc4ca6ec0e4f7dd0e0d')]


### When to use this shape

Reach for conditional interrupts the moment volume makes 100% human
review impractical, and you have *some* real signal to gate on -- a
dollar/risk threshold (as here), a model confidence score, or a business
rule (new customer vs. established one). This is the realistic, common
shape of HITL in deployed systems: the overwhelming majority of actions
never pause at all, and the review budget concentrates on the tail that
actually needs it.

### Common errors

- **Hardcoding the threshold in code instead of exposing it as
  configuration.** A threshold that requires a code change and redeploy
  to tune is a threshold nobody will actually tune -- `HITL_CONFIG` above
  is deliberately a plain, editable dict for this reason.
- **Picking a threshold with no data behind it.** A round number picked
  without looking at the actual distribution of refund amounts (or
  whatever the risk signal is) is a guess wearing the costume of a
  policy -- revisit it once real volume data exists.
- **Forgetting the low-risk path still needs auditing.** Skipping the
  *interrupt* doesn't mean skipping the *record* -- auto-approved actions
  still need to be logged somewhere reviewable after the fact, even
  though no human was in the loop at the moment they happened.

## Part 5 -- Full worked example: everything together

One small, self-contained agent combining a checkpointer, a conditional
`interrupt()`, and an editable resume payload -- run twice, once per path,
in the same session. This is deliberately architecture-agnostic: nothing
here is specific to any one topology in `multi_agent_architectures/` --
this exact node could sit inside a supervisor, a planner-executor, or a
single ReAct agent unchanged.

In [8]:
class FinalState(TypedDict):
    order_id: str
    extracted_amount: float
    final_amount: float
    approved: bool
    outcome: str


def extract(state: FinalState) -> dict:
    return {"extracted_amount": state["extracted_amount"]}


def maybe_review(state: FinalState) -> dict:
    if state["extracted_amount"] < HITL_CONFIG["auto_approve_below"]:
        return {"final_amount": state["extracted_amount"], "approved": True}
    corrected = interrupt({
        "action": "review_refund",
        "order_id": state["order_id"],
        "extracted_amount": state["extracted_amount"],
        "question": "Confirm or correct this refund amount, or resume with 0 to reject.",
    })
    return {"final_amount": corrected, "approved": corrected > 0}


def finalize(state: FinalState) -> dict:
    if state["approved"]:
        return {"outcome": issue_refund(state["order_id"], state["final_amount"])}
    return {"outcome": "Refund rejected by reviewer -- no action taken."}


final_builder = StateGraph(FinalState)
final_builder.add_node("extract", extract)
final_builder.add_node("maybe_review", maybe_review)
final_builder.add_node("finalize", finalize)
final_builder.add_edge(START, "extract")
final_builder.add_edge("extract", "maybe_review")
final_builder.add_edge("maybe_review", "finalize")
final_builder.add_edge("finalize", END)
final_graph = final_builder.compile(checkpointer=InMemorySaver())

# Path A -- low-risk, never pauses.
config_a = {"configurable": {"thread_id": "final-low"}}
result_a = final_graph.invoke({"order_id": "ORD-A", "extracted_amount": 8.00, "final_amount": 0.0, "approved": False, "outcome": ""}, config=config_a)
print("Path A (low-risk, $8.00):", result_a["outcome"])

# Path B -- high-risk, pauses, human corrects the amount, resumes.
config_b = {"configurable": {"thread_id": "final-high"}}
paused_b = final_graph.invoke({"order_id": "ORD-B", "extracted_amount": 5000.00, "final_amount": 0.0, "approved": False, "outcome": ""}, config=config_b)
print("Path B (high-risk, $5000.00 extracted) paused:", paused_b["__interrupt__"])

resumed_b = final_graph.invoke(Command(resume=500.00), config=config_b)
print("Path B resumed with human-corrected $500.00:", resumed_b["outcome"])


Path A (low-risk, $8.00): Refund of $8.00 issued for order ORD-A.
Path B (high-risk, $5000.00 extracted) paused: [Interrupt(value={'action': 'review_refund', 'order_id': 'ORD-B', 'extracted_amount': 5000.0, 'question': 'Confirm or correct this refund amount, or resume with 0 to reject.'}, id='74c6561afbfc99992f55f735fba160db')]
Path B resumed with human-corrected $500.00: Refund of $500.00 issued for order ORD-B.


**Expected output**: Path A completes in one call, no interrupt payload
ever printed -- the $8.00 refund never needed a human. Path B pauses
(extraction was off by 10x -- $5000 instead of $500), and the final
refund reflects the **human-corrected** $500.00, not the original
extraction. Two genuinely different runtime paths through the exact same
graph, decided entirely by one configurable threshold check.

## Revision summary

- **Three HITL mechanisms, not one**: static `interrupt_before=[node]`
  for an unconditional, payload-free gate; dynamic `interrupt()` inside a
  node for conditional gating or a custom review payload; a plain
  return-draft/separate-commit-call when there's no checkpointer or
  long-running graph process to pause in the first place.
- **`interrupt()` always needs a checkpointer** -- there's nowhere to
  persist the paused state without one, exactly as covered in
  `agent_memory_deepdive.ipynb`.
- **Side effects belong strictly after the interrupt point, in their own
  node if possible** -- resuming a node re-runs it from the top, so
  anything with a side effect placed before `interrupt()` in the same
  node fires again on every resume.
- **Approve/reject (Part 2) vs. edit (Part 3)** are different shapes for
  different situations: reject-and-retry is fine when redoing the whole
  step is cheap; editable state is worth it when the agent's output was
  mostly right and only needs a correction.
- **Conditional interrupts (Part 4)** are what most real production HITL
  actually looks like: a configurable threshold decides whether a given
  action pauses at all, so review effort concentrates on the risky tail
  instead of 100% of volume.
- HITL is a horizontal capability, not a topology -- everything in this
  notebook drops into any of the six patterns in
  `multi_agent_architectures/` unchanged.

## Explain like I'm 12

Imagine you're allowed to spend your family's money on groceries, but for
anything over $50 you have to text a parent first and wait for a
thumbs-up before you pay. For small stuff, you just buy it -- nobody
needs to be bothered. For the expensive stuff, you stop, show them
exactly what you're about to buy, and wait. And sometimes they don't just
say yes or no -- they text back "get the smaller size instead," and you
buy *that* instead of what you first picked. That's the whole notebook:
small stuff goes through automatically, big stuff pauses for a real
person, and that person can just say yes/no or actually change what
happens next.

## Explain for interview

"LangGraph offers human-in-the-loop through `interrupt()`, which pauses
graph execution at a specific point inside a node and persists that
paused state via the checkpointer, so it can be resumed later with
`Command(resume=...)` on the same `thread_id`. I'd choose static
`interrupt_before` for an unconditional checkpoint before a fixed node,
and dynamic `interrupt()` when the decision to pause is conditional --
gated on a risk score or dollar threshold -- or when I need to surface a
specific payload for a human to review or correct. The two failure modes
I watch for: placing side-effecting code before the `interrupt()` call
inside the same node, since LangGraph re-runs the whole node function on
resume, not just the code after the interrupt; and hardcoding the
gating threshold instead of exposing it as configuration, since that
threshold is exactly the kind of thing that needs tuning once real
volume/risk data exists. In production, the realistic shape is
conditional: most actions never pause, and review effort concentrates on
the tail that crosses a defined risk threshold."

## Glossary

- **`interrupt()`** -- pauses graph execution inside a node, returning
  control to the caller with a payload; resumed later via
  `Command(resume=...)` on the same `thread_id`.
- **`interrupt_before` / `interrupt_after`** -- static, node-level pause
  points set at `.compile()`, unconditional and payload-free.
- **`Command(resume=...)`** -- the object used to resume a paused graph,
  carrying the value that becomes `interrupt()`'s return value inside the
  paused node.
- **Approval gate** -- a binary (approve/reject) HITL checkpoint before an
  action.
- **Editable state** -- a HITL checkpoint where the human's response
  overwrites or corrects a value in state, rather than just branching on
  yes/no.
- **Conditional interrupt** -- an `interrupt()` call wrapped in a runtime
  check (threshold, confidence, rule), so only some fraction of runs
  actually pause.
- **Checkpointer** -- the LangGraph component (`InMemorySaver`,
  `SqliteSaver`, ...) that makes `interrupt()`/resume possible by
  persisting state per `thread_id` after every superstep.

## Checkpoint questions

1. **Q: Why can't `interrupt()` work without a checkpointer?**
   A: Pausing mid-execution only means something if the paused state is
   saved somewhere -- the checkpointer is what persists it, keyed by
   `thread_id`, so `Command(resume=...)` has something to resume from.

2. **Q: When would you use static `interrupt_before=[node]` instead of a
   dynamic `interrupt()` call inside the node?**
   A: When the pause should be unconditional and payload-free -- always
   stop before this node, no custom message needed, no runtime decision
   about whether to pause at all.

3. **Q: What is the single most damaging HITL bug this notebook calls
   out, and why does it happen?**
   A: Placing side-effecting code *before* the `interrupt()` call inside
   the same node -- because resuming a node re-runs the entire function
   from the top, that side effect fires again on every resume, not just
   once.

4. **Q: What's the practical difference between Part 2's approval gate
   and Part 3's editable state?**
   A: The gate only lets a human say yes or no, discarding a mostly-right
   result on rejection; editable state lets the human correct a specific
   value, which then flows downstream exactly as if the agent had
   produced it.

5. **Q: Why does Part 4 make the auto-approve threshold a config dict
   instead of a hardcoded number?**
   A: So it can be tuned without a code change/redeploy -- a threshold
   that requires that to adjust is one nobody will actually keep current
   as real data changes.

6. **Q: In the Part 5 combined example, why does Path A never produce an
   `__interrupt__` key in its result?**
   A: Its amount is below `HITL_CONFIG["auto_approve_below"]`, so
   `maybe_review` never calls `interrupt()` at all on that path -- the
   graph runs straight through in one `.invoke()` call.

7. **Q: If a human resumes with an invalid or nonsensical edited value
   (e.g. a negative refund amount), what stops that from being processed?**
   A: Nothing, unless the code after `interrupt()` explicitly validates
   it -- HITL is not a substitute for normal input validation on
   human-provided data, which is just as untrusted as any other input.

8. **Q: Why is this notebook not numbered inside `multi_agent_architectures/`?**
   A: Because HITL is a horizontal capability that applies to any
   topology in that series unchanged, not a topology in its own right --
   the same reasoning that keeps memory and context engineering as
   separate, architecture-agnostic notebooks.

9. **Q: What would happen if you called `Command(resume=...)` on a
   `thread_id` that was never actually paused (no prior `interrupt()`
   call on that thread)?**
   A: There's no paused state to resume from that `thread_id`, so it
   would not behave like a meaningful resume -- treat `Command(resume=...)`
   as only valid immediately after a matching `interrupt()` pause on the
   same thread.

10. **Q: Give one production scenario where an *unconditional* gate
    (Part 2) is more appropriate than a *conditional* one (Part 4).**
    A: Any action where even a single unreviewed wrong instance is
    unacceptable regardless of amount/confidence -- e.g. deleting a
    production database, or an action with legal/regulatory sign-off
    requirements -- volume-based thresholds don't apply when the
    tolerance for an unreviewed mistake is zero.